# 05 — MLP models

MLP classifier training and evaluation. Uses `src.models.mlp` and `src.training.evaluation`.

In [ ]:
# Path fix: use this repo's src
from pathlib import Path
import sys
import logging
import numpy as np
import torch
PROJECT_ROOT = Path.cwd().resolve().parents[0]
project_root_str = str(PROJECT_ROOT)
if project_root_str in sys.path:
    sys.path.remove(project_root_str)
sys.path.insert(0, project_root_str)
for name in list(sys.modules.keys()):
    if name == "src" or name.startswith("src."):
        del sys.modules[name]

from src.utils import SEED, set_global_seed
from src.utils.paths import get_reduced_dir, get_splits_dir, get_labels_dir, get_experiment_dir, get_experiment_output_dir, ensure_dir
from src.data import CLASS_NAMES, load_labels
from src.data.splits import load_splits
from src.models.mlp import MLPClassifier
from src.training.evaluation import compute_metrics, compute_per_class_metrics, save_metrics, save_confusion_matrix

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

# Toggles: which feature pipelines to run (set to True/False)
RUN_AUTOENCODER = True  # A1: autoencoder-reduced features
RUN_POOLED = True       # B1: pooled (kernel-informed) features

# MLP / training params (match A1 and B1 configs)
EPOCHS = 20
BATCH_SIZE = 256
LR = 1e-3
WEIGHT_DECAY = 1e-4
HIDDEN_DIMS = [512, 256]
DROPOUT = 0.1
NUM_CLASSES = len(CLASS_NAMES)  # 3 conditions + Sinus Rhythm (none)
print("MLP: epochs=%s batch=%s lr=%s | %d classes: %s" % (EPOCHS, BATCH_SIZE, LR, NUM_CLASSES, CLASS_NAMES))
print("Run autoencoder (A1): %s | Run pooled (B1): %s" % (RUN_AUTOENCODER, RUN_POOLED))

MLP: epochs=20 batch=256 lr=0.001 | 4 classes: ['AF', 'SVT', 'Sinus Brady', 'Sinus Rhythm']
Run autoencoder (A1): False | Run pooled (B1): True


In [8]:
# Load splits and labels (same order as saved train/test/val .npy)
print("Loading splits and labels...")
idx_train, idx_val, idx_test = load_splits()
try:
    y_full = load_labels()
except FileNotFoundError:
    print("labels.npy not found; loading from raw dataset...")
    from src.data import load_raw_dataset
    _, y_full = load_raw_dataset()
if y_full.ndim == 2:
    y_idx = np.argmax(y_full, axis=1)
else:
    y_idx = y_full
y_train = y_idx[idx_train]
y_test = y_idx[idx_test]
y_val = y_idx[idx_val] if len(idx_val) > 0 else None
print("y_train %s y_test %s" % (y_train.shape, y_test.shape))

Loading splits and labels...
y_train (36120,) y_test (6773,)


In [9]:
def train_mlp_and_evaluate(X_train, X_val, X_test, y_train, y_val, y_test, input_dim, condition_name):
    """Train MLP on train set; evaluate on validation (if present) then on test. Save checkpoint and metrics."""
    set_global_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    exp_dir = ensure_dir(get_experiment_dir(condition_name))
    ckpt_dir = ensure_dir(get_experiment_output_dir(condition_name, checkpoints=True))

    model = MLPClassifier(
        input_dim=input_dim,
        num_classes=NUM_CLASSES,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
    ).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    n_train = len(X_train)

    for ep in range(EPOCHS):
        model.train()
        perm = np.random.permutation(n_train)
        total_loss = 0.0
        n_b = 0
        for start in range(0, n_train, BATCH_SIZE):
            idx = perm[start : start + BATCH_SIZE]
            bx = torch.from_numpy(X_train[idx]).float().to(device)
            by = torch.from_numpy(y_train[idx]).long().to(device)
            logits = model(bx)
            loss = torch.nn.functional.cross_entropy(logits, by)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total_loss += loss.item()
            n_b += 1
        if (ep + 1) % 5 == 0 or ep == 0:
            print("  Epoch %d loss %.4f" % (ep + 1, total_loss / max(n_b, 1)))

    torch.save(model.state_dict(), ckpt_dir / "mlp.pt")
    print("  Saved %s" % (ckpt_dir / "mlp.pt"))

    model.eval()
    all_metrics = {}
    labels_list = list(range(NUM_CLASSES))
    with torch.no_grad():
        if X_val is not None and y_val is not None and len(y_val) > 0:
            logits_val = model(torch.from_numpy(X_val).float().to(device))
            y_pred_val = logits_val.argmax(dim=1).cpu().numpy()
            metrics_val = compute_metrics(y_val, y_pred_val, task="multiclass", labels=labels_list)
            all_metrics["weighted_f1_val"] = metrics_val["weighted_f1"]
            per_class_val = compute_per_class_metrics(y_val, y_pred_val, labels=labels_list, target_names=CLASS_NAMES)
            all_metrics["per_class_val"] = per_class_val
            print("  Validation weighted F1: %.4f" % metrics_val["weighted_f1"])
            print("  Validation per-class:")
            for p in per_class_val:
                print("    %s: P=%.3f R=%.3f F1=%.3f support=%d" % (p["class"], p["precision"], p["recall"], p["f1"], p["support"]))
        logits_test = model(torch.from_numpy(X_test).float().to(device))
        y_pred_test = logits_test.argmax(dim=1).cpu().numpy()
    metrics_test = compute_metrics(y_test, y_pred_test, task="multiclass", labels=labels_list)
    per_class_test = compute_per_class_metrics(y_test, y_pred_test, labels=labels_list, target_names=CLASS_NAMES)
    all_metrics["weighted_f1"] = metrics_test["weighted_f1"]
    all_metrics["confusion_matrix"] = metrics_test.get("confusion_matrix", [])
    all_metrics["per_class"] = per_class_test
    save_metrics(all_metrics, exp_dir / "metrics.json")
    cm = np.array(metrics_test["confusion_matrix"])
    save_confusion_matrix(cm, exp_dir / "confusion_matrix.npy", path_png=exp_dir / "confusion_matrix.png", class_names=CLASS_NAMES)
    print("  Test weighted F1: %.4f" % metrics_test["weighted_f1"])
    print("  Test per-class:")
    for p in per_class_test:
        print("    %s: P=%.3f R=%.3f F1=%.3f support=%d" % (p["class"], p["precision"], p["recall"], p["f1"], p["support"]))
    return all_metrics

In [10]:
# A1: Autoencoder + MLP (load autoencoder-reduced features)
metrics_a1 = None
if RUN_AUTOENCODER:
    print("=== A1: Autoencoder + MLP ===")
    red_dir = get_reduced_dir() / "autoencoder"
    X_train_a1 = np.load(red_dir / "train.npy").astype(np.float32)
    X_test_a1 = np.load(red_dir / "test.npy").astype(np.float32)
    if (red_dir / "val.npy").exists():
        X_val_a1 = np.load(red_dir / "val.npy").astype(np.float32)
    else:
        X_val_a1 = None
    print("Loaded autoencoder features: train %s test %s (input_dim=%d)" % (X_train_a1.shape, X_test_a1.shape, X_train_a1.shape[1]))
    metrics_a1 = train_mlp_and_evaluate(X_train_a1, X_val_a1, X_test_a1, y_train, y_val, y_test, X_train_a1.shape[1], "A1_autoencoder_mlp")
else:
    print("Skipping A1 (autoencoder + MLP) — RUN_AUTOENCODER=False")

Skipping A1 (autoencoder + MLP) — RUN_AUTOENCODER=False


In [ ]:
# B1: Pooling + MLP (load pooled features)
metrics_b1 = None
if RUN_POOLED:
    print("=== B1: Pooling + MLP ===")
    red_dir = get_reduced_dir() / "pooled"
    X_train_b1 = np.load(red_dir / "train.npy").astype(np.float32)
    X_test_b1 = np.load(red_dir / "test.npy").astype(np.float32)
    if (red_dir / "val.npy").exists():
        X_val_b1 = np.load(red_dir / "val.npy").astype(np.float32)
    else:
        X_val_b1 = None
    print("Loaded pooled features: train %s test %s (input_dim=%d)" % (X_train_b1.shape, X_test_b1.shape, X_train_b1.shape[1]))
    metrics_b1 = train_mlp_and_evaluate(X_train_b1, X_val_b1, X_test_b1, y_train, y_val, y_test, X_train_b1.shape[1], "B1_pooling_mlp")
else:
    print("Skipping B1 (pooling + MLP) — RUN_POOLED=False")

2026-03-09 16:04:14,146 | INFO | Global seed set to 0


=== B1: Pooling + MLP ===
Loaded pooled features: train (36120, 176) test (6773, 176) (input_dim=176)
  Epoch 1 loss 0.4930
  Epoch 5 loss 0.2508
  Epoch 10 loss 0.1975


In [ ]:
# Summary: overall and per-class (test set)
print("=== Summary ===")
if metrics_a1 is not None:
    print("A1 (autoencoder + MLP) test weighted F1: %.4f" % metrics_a1["weighted_f1"])
if metrics_b1 is not None:
    print("B1 (pooling + MLP) test weighted F1: %.4f" % metrics_b1["weighted_f1"])
if metrics_a1 is not None or metrics_b1 is not None:
    print("\nPer-class (test set):")
    n_cols = (3 if metrics_a1 else 0) + (3 if metrics_b1 else 0)
    hdr = "%-12s | " % "Class" + " ".join("%8s" % h for h in (["A1_P", "A1_R", "A1_F1"] if metrics_a1 else []) + (["B1_P", "B1_R", "B1_F1"] if metrics_b1 else []))
    print(hdr)
    print("-" * len(hdr))
    for i, name in enumerate(CLASS_NAMES):
        row = [name]
        if metrics_a1:
            a1 = metrics_a1.get("per_class", [])[i] if "per_class" in metrics_a1 else {}
            row.extend([a1.get("precision", 0), a1.get("recall", 0), a1.get("f1", 0)])
        if metrics_b1:
            b1 = metrics_b1.get("per_class", [])[i] if "per_class" in metrics_b1 else {}
            row.extend([b1.get("precision", 0), b1.get("recall", 0), b1.get("f1", 0)])
        print("%-12s | " % row[0] + " ".join("%8.3f" % v for v in row[1:]))
else:
    print("No runs selected (set RUN_AUTOENCODER and/or RUN_POOLED to True).")

=== Summary ===
B1 (pooling + MLP) test weighted F1: 0.9088

Per-class (test set):
Class        |     B1_P     B1_R    B1_F1
-----------------------------------------
AF           |    0.913    0.898    0.906
SVT          |    0.625    0.525    0.570
Sinus Brady  |    0.947    0.971    0.959
Sinus Rhythm |    0.854    0.864    0.859
